In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.evaluate_models import eval_reg

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

print("Session 30 imports successful")

Session 30 imports successful


In [2]:
df = pd.read_csv(
    "../data/student-mat.csv",
    sep=";"
)

df_encoded = pd.get_dummies(
    df,
    drop_first=True
)

X_full = df_encoded.drop(
    columns=["G3"]
).copy()

y = df_encoded["G3"].copy()

print("Dataset loaded.")
print("X_full shape:", X_full.shape)
print("y shape:", y.shape)

Dataset loaded.
X_full shape: (395, 41)
y shape: (395,)


In [3]:
Xtr_f, Xte_f, ytr, yte = train_test_split(
    X_full,
    y,
    test_size=0.20,
    random_state=42
)

print("Train/Test split complete.")
print("Xtr_f:", Xtr_f.shape)
print("Xte_f:", Xte_f.shape)

Train/Test split complete.
Xtr_f: (316, 41)
Xte_f: (79, 41)


In [4]:
mlp_pipeline = Pipeline([
    ("standardscaler", StandardScaler()),
    (
        "mlpregressor",
        MLPRegressor(
            hidden_layer_sizes=(64, 32),
            max_iter=1000,
            random_state=42
        )
    )
])

print("MLP pipeline created.")

MLP pipeline created.


In [5]:
print("Pipeline steps:")

for step_name, step_object in mlp_pipeline.steps:
    print(step_name, "->", step_object)

Pipeline steps:
standardscaler -> StandardScaler()
mlpregressor -> MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42)


In [6]:
mlp_pipeline.fit(
    Xtr_f,
    ytr
)

print("MLP training complete.")

MLP training complete.


In [7]:
mlp_predictions = mlp_pipeline.predict(
    Xte_f
)

print(
    "Number of predictions:",
    len(mlp_predictions)
)

print(
    "First five predictions:"
)

print(
    mlp_predictions[:5]
)

Number of predictions: 79
First five predictions:
[ 7.41737682 10.89638306  2.39121933 10.61202244 11.15627049]


In [8]:
mlp_test_metrics = eval_reg(
    yte,
    mlp_predictions
)

print("MLP Test Results")
print(mlp_test_metrics)

MLP Test Results
{'MAE': 2.020337290519022, 'RMSE': 2.784599125864401, 'R2': 0.6218495773067232}


In [9]:
import pandas as pd

mlp_result_row = pd.DataFrame([
    {
        "Model": "MLP Regressor",
        "Scenario": "Full Information",
        "Scaling": "StandardScaler",
        "MAE": mlp_test_metrics["MAE"],
        "RMSE": mlp_test_metrics["RMSE"],
        "R2": mlp_test_metrics["R2"]
    }
])

mlp_result_row.round(4)

,Model,Scenario,Scaling,MAE,RMSE,R2
0,MLP Regressor,Full Information,StandardScaler,2.0203,2.7846,0.6218


In [10]:
comparison_table = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Scenario": "Full Information",
        "Scaling": "None",
        "MAE": 1.1706329113924052,
        "RMSE": 1.97473322451841,
        "R2": 0.8098238322966482
    },
    {
        "Model": "Gradient Boosting",
        "Scenario": "Full Information",
        "Scaling": "None",
        "MAE": 1.267622292850229,
        "RMSE": 2.154159988807915,
        "R2": 0.7736944862054645
    },
    {
        "Model": "Extra Trees",
        "Scenario": "Full Information",
        "Scaling": "None",
        "MAE": 1.3311814345991562,
        "RMSE": 2.2644825354383293,
        "R2": 0.7499210274296113
    },
    {
        "Model": "Decision Tree",
        "Scenario": "Full Information",
        "Scaling": "None",
        "MAE": 1.357218203737191,
        "RMSE": 2.467248497769575,
        "R2": 0.7031308891822728
    },
    {
        "Model": "SVR",
        "Scenario": "Full Information",
        "Scaling": "StandardScaler",
        "MAE": 1.8366678536589514,
        "RMSE": 2.7260297218073055,
        "R2": 0.6375898115704413
    },
    {
        "Model": "MLP Regressor",
        "Scenario": "Full Information",
        "Scaling": "StandardScaler",
        "MAE": mlp_test_metrics["MAE"],
        "RMSE": mlp_test_metrics["RMSE"],
        "R2": mlp_test_metrics["R2"]
    },
    {
        "Model": "KNN",
        "Scenario": "Full Information",
        "Scaling": "StandardScaler",
        "MAE": 2.5848101265822785,
        "RMSE": 3.3926950565642415,
        "R2": 0.4386562685587473
    }
])

comparison_table = (
    comparison_table
    .sort_values(
        by="RMSE",
        ascending=True
    )
    .reset_index(drop=True)
)

comparison_table.index += 1

comparison_table

,Model,Scenario,Scaling,MAE,RMSE,R2
1,Random Forest,Full Information,None,1.170633,1.974733,0.809824
2,Gradient Boosting,Full Information,None,1.267622,2.154160,0.773694
3,Extra Trees,Full Information,None,1.331181,2.264483,0.749921
4,Decision Tree,Full Information,None,1.357218,2.467248,0.703131
5,SVR,Full Information,StandardScaler,1.836668,2.726030,0.637590
6,MLP Regressor,Full Information,StandardScaler,2.020337,2.784599,0.621850
7,KNN,Full Information,StandardScaler,2.584810,3.392695,0.438656
